# BuildingsBench-faithful load forecasting study — canonical models, one notebook

Trains + evaluates **21 models** (3 official BuildingsBench persistence baselines; LSTM, GRU;
XGBoost, LightGBM; 7 canonical published Transformer/linear architectures — PatchTST,
iTransformer, TimeXer, DLinear, Informer, Autoformer, Crossformer; the paper's **own**
pretrained model family, Transformer-S/M/L (Gaussian); and 4 novel architectures grounded in
post-BuildingsBench literature — TFTLite, xLSTM, SpectraMix, Mamba, see below) across
**2 conditions** (no-weather / +weather), on a compute-bounded subsample of the official
Buildings-900K pretraining corpus, evaluated on **both** the official simulated test set and 7
real-building datasets.

All logic lives in the `bblab/` package (cloned from this repo below) — this notebook is a thin,
readable run log. Preprocessing (Box-Cox load transform, per-channel weather StandardScaler,
linearly-scaled calendar features, the confirmed 7-channel weather order) is sourced directly
from the official [BuildingsBench repo](https://github.com/NatLabRockies/BuildingsBench) and the
public `oedi-data-lake` S3 bucket it publishes to — not a from-scratch reimplementation.

### What changed vs. the previous draft of this study
- Dropped 5 homegrown architectures in favor of the field's standard published baselines, and
  added the paper's own Transformer-S/M/L (Gaussian) architecture as a direct comparison point
  (faithful port of `buildings_bench/models/transformers.py`: encoder-decoder `nn.Transformer`,
  teacher forcing at train time, autoregressive greedy decoding at inference — same shared
  train/eval loop as every other model here, no separate code path to maintain).
- Fixed: unnormalized calendar features in real-building eval, a crashed results cell
  (`PERBUILDING_REAL_DIR` NameError), single-fixed-window validation, a silently zero-filled
  corrupt training building, and missing multiple-comparison correction on significance tests.
- Validation now matches the paper's actual split (temporal holdout of the last 2 weeks of the
  year, amy2018-release buildings only) instead of an arbitrary building-level carve-out from
  the training subsample — see the Data section below.
- Calendar encoding corrected to match the paper exactly (linear-scaled day-of-year/day-of-week/
  hour-of-day, **not** sin/cos).
- Persistence baseline is now the paper's actual 3 variants (Average/CopyLastDay/CopyLastWeek),
  not a single custom "ensemble."
- Real-building evaluation runs a headline **no-weather** track (every real building) plus an
  optional **+weather** track (temperature+humidity, wherever the public bucket has it — no
  external fetching, so buildings without published weather simply aren't in that track).

### Novel architectures (`config.NOVEL_MODELS`)
Four new models grounded in post-2023 forecasting literature, aiming to beat the canonical
baselines above rather than just replicate them. All share the same `(yh, exo) -> (mu_n,
raw_scale, mean, std)` interface as every other model, so they drop straight into the existing
train/eval loop with no special-casing.
- **`tftlite`** — a compact Temporal Fusion Transformer (Lim et al. 2021): per-timestep Variable
  Selection Networks (Gated Residual Network-weighted softmax over each input channel) feeding
  an LSTM encoder-decoder, a static enrichment GRN, then causal self-attention over the full
  history+horizon. Tests whether learned per-feature gating beats treating all exogenous inputs
  uniformly.
- **`xlstm`** — Beck et al. 2024's xLSTM sLSTM cell (exponential input/forget gates with a
  log-space stabilizer state, replacing the sigmoid gates of a vanilla LSTM) applied over 24h
  patches (7 patches per 168h window, not raw hourly steps — keeps the sequential recurrence
  affordable). Motivated by a 2024 building-heat-load benchmark where xLSTM outperformed both a
  Transformer and TFT.
- **`spectramix`** — TSMixer-style time-mixing + feature-mixing MLP blocks (Chen et al. 2023)
  over daily patches, blended with a FITS-style (Xu et al. 2024) complex-linear frequency-domain
  extrapolation branch (learned weights map the historical rFFT spectrum directly onto the
  extended-horizon spectrum), plus a differentiable heating/cooling degree-day term (learned
  balance points, ReLU degree-hours -> load correction) when weather is available. Tests whether
  a lightweight mixing+frequency approach with an explicit physical prior competes with
  attention-heavy models at a fraction of the parameter count.
- **`mamba`** — a from-scratch selective state-space scan (Gu & Dao 2023's S6: input-dependent
  discretization step size gating a diagonal linear recurrence) over daily patches, gated MLP
  around it in the Mamba block style. Tests the current "attention vs. selective-SSM" debate on
  this specific forecasting task.

### A note on compute
Transformer-L matches the paper's largest config (12+12 encoder/decoder layers, d_model=768,
~160M+ params) and is meaningfully heavier per epoch than everything else here. If you want to
skip it, run `config.ALL_MODELS.remove("transformer_l")` right after cell 2 (before cell 6) —
resumable checkpointing means it's otherwise fine to just let it span multiple Colab sessions.

### Resumability
Every long-running cell below writes to Google Drive and skips already-completed
(model, condition) pairs on re-run — safe to re-run this notebook after a Colab disconnect.


In [1]:
# === 1) Mount Drive, clone repo, install deps, GPU check ===
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = "https://github.com/nm-quan/Building-Bench-Future-Works"
REPO_DIR = "/content/Building-Bench-Future-Works"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > A100."
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# === 2) Imports + config ===
from bblab import config, data, models, train, metrics, weather_real
import torch, numpy as np, pandas as pd, pickle, os, csv, time

paths = config.Paths().makedirs()
print(f"BENCH={paths.BENCH}")
print(f"models ({len(config.ALL_MODELS)}): {config.ALL_MODELS}")
print(f"L={config.L} H={config.H}  EPOCHS={config.EPOCHS} PATIENCE={config.PATIENCE}  N_TRAIN_BUILDINGS={config.N_TRAIN_BUILDINGS}")


BENCH=/content/drive/MyDrive/quick/bench
models (21): ['persistence_avg', 'persistence_last_day', 'persistence_last_week', 'lstm', 'gru', 'patchtst', 'itransformer', 'timexer', 'dlinear', 'informer', 'autoformer', 'crossformer', 'transformer_s', 'transformer_m', 'transformer_l', 'xgboost', 'lightgbm', 'tftlite', 'xlstm', 'spectramix', 'mamba']
L=168 H=24  EPOCHS=200 PATIENCE=20  N_TRAIN_BUILDINGS=20000


## Data

Sources the official BuildingsBench transforms + a compute-bounded (~20k building) subsample of
Buildings-900K, the official held-out Buildings-900K-test simulated test set, and the 7
real-building datasets — all directly from the public `oedi-data-lake` S3 bucket. Cached to Drive
under `paths.BENCH` so re-running this notebook doesn't re-download/re-process from scratch.

**Validation** matches the official split exactly (`scripts/data_generation/create_index_files.py`):
a temporal holdout of the last 336h (2 weeks) of the year, on the amy2018-release buildings only
(a real chronological year -- unlike the tmy3 releases, which splice together a synthetic "typical"
year with no genuine chronological tail, and which the paper's own script never validates on).
`train.train()` samples training windows so they never cross into that holdout range for amy2018
buildings, and validates only within it -- not a same-corpus building-level carve-out.


In [3]:
# === 3) Official BuildingsBench transforms (Box-Cox load, per-channel weather StandardScaler) ===
transforms = data.download_official_transforms(paths)
load_transform = transforms["load"]
weather_transforms = transforms["weather"]
print("weather channels (confirmed order):", list(weather_transforms.keys()))


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


weather channels (confirmed order): ['temperature', 'humidity', 'wind_speed', 'wind_direction', 'global_horizontal_radiation', 'direct_normal_radiation', 'diffuse_horizontal_radiation']


In [4]:
# === 4) Build the training subsample (official weekly-sampled index file) + official simulated test set ===
torch.manual_seed(config.SEED); np.random.seed(config.SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

print("building train cache (first run downloads from S3; cached on Drive afterward)...")
TR_raw = data.build_train_cache(paths, transforms, n_buildings=config.N_TRAIN_BUILDINGS, seed=config.SEED)
print(f"  TRAIN  N={TR_raw['N']}  T={TR_raw['T']}h  n_time={TR_raw['Ft']} n_weather={TR_raw['Fw']}")

print("building simulated test cache (official Buildings-900K-test, held out from training)...")
TE_SIM_raw = data.build_sim_test_cache(paths, transforms)
print(f"  SIM TEST  N={TE_SIM_raw['N']}  T={TE_SIM_raw['T']}h")

TR = train.to_device_cache(TR_raw, DEV)
TE_SIM = train.to_device_cache(TE_SIM_raw, DEV)


building train cache (first run downloads from S3; cached on Drive afterward)...
  [cache] found existing /content/drive/MyDrive/quick/bench/train_20k.npz, loading
  TRAIN  N=20000  T=8759h  n_time=3 n_weather=7
building simulated test cache (official Buildings-900K-test, held out from training)...
  [cache] found existing /content/drive/MyDrive/quick/bench/sim_test.npz, loading
  SIM TEST  N=2928  T=8759h


In [5]:
# === 5) Load the 7 real-building datasets directly from the public S3 bucket (no separate data-prep notebook needed) ===
TE_REAL = data.load_real_buildings(paths)
n_com = sum(1 for b in TE_REAL if b["building_type"] > 0)
n_res = sum(1 for b in TE_REAL if b["building_type"] < 0)
print(f"REAL TEST  N={len(TE_REAL)}  com={n_com}  res={n_res}")


  [real] BDG-2: 1078 buildings from 8 files, dropped 1 empty/unparseable files
  [real] Borealis: 15 buildings from 15 files
  [real] Electricity: 1159 buildings from 4 files


KeyboardInterrupt: 

## Train all models x both conditions (resumable)

`ALL_MODELS` = 3 persistence baselines (closed-form, 0 params) + LSTM/GRU/DLinear/PatchTST/
iTransformer/TimeXer/Informer/Autoformer/Crossformer (trained) + the paper's own
Transformer-S/M/L (Gaussian) (trained, teacher forcing) + XGBoost/LightGBM (fit) +
TFTLite/xLSTM/SpectraMix/Mamba (trained, `config.NOVEL_MODELS`). Skips any (model, condition)
pair already present in `sim_results.csv` — safe to re-run after a disconnect. This is the
long-running cell; expect several hours (more with Transformer-L included) for the full sweep
on an A100.


In [ ]:
# === 6) Train all models x both conditions (resumable) ===
train.run_training_sweep(config.ALL_MODELS, config.CONDITIONS, TR, TE_SIM, DEV, load_transform, paths,
                          reset=False, log=True)


  done 17s  Com NRMSE=101.764  Res NRMSE=125.965  (med 62.479/96.332)

  done 17s  Com NRMSE=101.764  Res NRMSE=125.965  (med 62.479/96.332)

  done 13s  Com NRMSE=131.842  Res NRMSE=159.999  (med 81.023/123.994)

  done 13s  Com NRMSE=131.842  Res NRMSE=159.999  (med 81.023/123.994)

  done 13s  Com NRMSE=131.773  Res NRMSE=160.368  (med 80.92/124.126)

  done 13s  Com NRMSE=131.773  Res NRMSE=160.368  (med 80.92/124.126)

reuse checkpoint lstm/A (no retraining)
  done 22s  Com NRMSE=98.136  Res NRMSE=124.407  (med 59.899/94.509)

reuse checkpoint lstm/B (no retraining)
  done 24s  Com NRMSE=90.447  Res NRMSE=108.64  (med 54.001/86.348)

reuse checkpoint gru/A (no retraining)
  done 22s  Com NRMSE=98.097  Res NRMSE=124.945  (med 59.888/94.884)

reuse checkpoint gru/B (no retraining)
  done 23s  Com NRMSE=90.254  Res NRMSE=108.189  (med 53.696/85.951)

  patchtst/A: checkpoint is from an older architecture size -- retraining
train patchtst/A (7,557,936p, amp=True)...
    ep000 val=1.41

## Simulated-test results

Median [95% bootstrap CI] NRMSE per model/condition, paired Wilcoxon significance vs. the
`persistence_avg` baseline (Holm-Bonferroni corrected across the model set — the earlier draft
of this study ran these tests uncorrected), plus a Friedman + Nemenyi comparison across the
*full* model set (which models actually differ, with proper multiple-comparison control).


In [ ]:
# === 7) Simulated-test results: median [95% bootstrap CI], Holm-Bonferroni-corrected significance vs persistence_avg ===
def load_pb(path):
    try:
        return pickle.load(open(path, "rb"))
    except FileNotFoundError:
        return None

BASELINE = config.BASELINE_MODEL
for cond in ["A", "B"]:
    print(f"-- Condition {cond} ({'no weather' if cond == 'A' else '+weather'}) --")
    base_pb = load_pb(f"{paths.PERBUILDING_SIM_DIR}/{BASELINE}_{cond}.pkl")
    pvals = {}
    for name in config.ALL_MODELS:
        pb = load_pb(f"{paths.PERBUILDING_SIM_DIR}/{name}_{cond}.pkl")
        if pb is None:
            continue
        vals = pb["_nrmse"][~pb["_is_res"]]
        med, lo, hi = metrics.bootstrap_median_ci(vals)
        note = ""
        if base_pb is not None and name != BASELINE:
            bvals = base_pb["_nrmse"][~base_pb["_is_res"]]
            p, better = metrics.paired_significance(vals, bvals)
            pvals[name] = p
            if better is True:
                note = f"  beats {BASELINE} (p={p:.1e}, uncorrected)"
            elif better is False and p == p and p < 0.05:
                note = f"  WORSE than {BASELINE} (p={p:.1e}, uncorrected)"
        print(f"  {name:24s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{note}")
    corrected = metrics.holm_bonferroni(pvals)
    winners = [k for k, (p, sig) in corrected.items() if sig]
    print(f"  Holm-Bonferroni-corrected significant wins vs {BASELINE}: {winners}\n")


In [ ]:
# === 7b) Friedman + Nemenyi across the full model set (which models differ, with multiple-comparison control) ===
for cond in ["A", "B"]:
    pbs = {name: load_pb(f"{paths.PERBUILDING_SIM_DIR}/{name}_{cond}.pkl") for name in config.ALL_MODELS}
    pbs = {k: v for k, v in pbs.items() if v is not None}
    names = list(pbs.keys())
    arrs = [pbs[n]["_nrmse"][~pbs[n]["_is_res"]] for n in names]
    min_len = min(len(a) for a in arrs)
    mat = np.stack([a[:min_len] for a in arrs], axis=1)
    out = metrics.friedman_nemenyi(mat, names)
    ranked = sorted(out["avg_ranks"].items(), key=lambda kv: kv[1])
    print(f"cond {cond}: Friedman p={out['friedman_p']:.2e}  Nemenyi CD={out['nemenyi_critical_difference']:.3f}")
    print("  avg ranks (lower = better):", {k: round(v, 2) for k, v in ranked})
    print()


## Real-building evaluation

Two tracks:
1. **No-weather (headline)** — every real building, using the no-weather checkpoints. This is
   the primary real-world generalization result.
2. **+weather (temperature/humidity)** — the subset of real buildings whose dataset has published
   era5 weather on the public bucket (BuildingsBench's own real-eval only uses temperature; we
   include humidity too since it's already published alongside it). Buildings without published
   weather simply aren't in this track — no external fetching, no filled-in data.


In [ ]:
# === 8) Real-building evaluation: no-weather checkpoints (headline table, every real building) ===
RFIELDS = ["model", "com_nrmse", "res_nrmse", "com_nmae", "res_nmae", "com_crps", "res_crps", "n_buildings", "sec"]
if not os.path.exists(paths.REAL_CSV):
    csv.DictWriter(open(paths.REAL_CSV, "w", newline=""), RFIELDS).writeheader()
rdone = set()
if os.path.getsize(paths.REAL_CSV) > 50:
    rdone = {r.model for _, r in pd.read_csv(paths.REAL_CSV).iterrows()}

for name in config.ALL_MODELS:
    if name in rdone:
        print(f"skip {name} (real-eval)")
        continue
    t0 = time.time()
    try:
        if name in ("xgboost", "lightgbm"):
            mdl, sigma = pickle.load(open(f"{paths.WEIGHTS_DIR}/{name}_A.pkl", "rb"))
            res = train.evaluate_real_tree(mdl, sigma, TE_REAL, load_transform, weather_cache=None, stride=24)
        else:
            base = models.build(name, L=config.L, H=config.H, n_time=TR["Ft"], n_weather=TR["Fw"],
                                 use_weather=False, **models.MODEL_KW.get(name, {})).to(DEV)
            if models.count_params(base) > 0:
                sd_path = f"{paths.WEIGHTS_DIR}/{name}_A.pt"
                if not os.path.exists(sd_path):
                    print(f"  {name}: no no-weather checkpoint yet, skip")
                    continue
                base.load_state_dict(torch.load(sd_path, map_location=DEV))
            print(f"real-eval {name}...", flush=True)
            res = train.evaluate_real(base, TE_REAL, DEV, load_transform, weather_cache=None,
                                       amp=config.USE_AMP and name not in config.RNN_MODELS, stride=24)
        os.makedirs(paths.PERBUILDING_REAL_DIR, exist_ok=True)
        pickle.dump({k: res[k] for k in ("_nrmse", "_nmae", "_crps", "_is_res") if k in res},
                    open(f"{paths.PERBUILDING_REAL_DIR}/{name}.pkl", "wb"))
        row = {"model": name,
               "com_nrmse": round(res.get("Com NRMSE", float("nan")), 3),
               "res_nrmse": round(res.get("Res NRMSE", float("nan")), 3),
               "com_nmae": round(res.get("Com NMAE", float("nan")), 3) if res.get("Com NMAE") == res.get("Com NMAE") else "",
               "res_nmae": round(res.get("Res NMAE", float("nan")), 3) if res.get("Res NMAE") == res.get("Res NMAE") else "",
               "com_crps": round(res.get("Com CRPS", float("nan")), 3) if res.get("Com CRPS") == res.get("Com CRPS") else "",
               "res_crps": round(res.get("Res CRPS", float("nan")), 3) if res.get("Res CRPS") == res.get("Res CRPS") else "",
               "n_buildings": res.get("n_buildings", 0), "sec": round(time.time() - t0)}
        csv.DictWriter(open(paths.REAL_CSV, "a", newline=""), RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
print("DONE ->", paths.REAL_CSV)


In [ ]:
# === 9) Real-building evaluation: +weather (temperature/humidity, wherever published) ===
RWFIELDS = ["model", "com_nrmse", "res_nrmse", "com_nmae", "res_nmae", "com_crps", "res_crps",
            "n_buildings", "n_weather_matched", "sec"]
if not os.path.exists(paths.REAL_WEATHER_TEMP_CSV):
    csv.DictWriter(open(paths.REAL_WEATHER_TEMP_CSV, "w", newline=""), RWFIELDS).writeheader()
rwdone = set()
if os.path.getsize(paths.REAL_WEATHER_TEMP_CSV) > 50:
    rwdone = {r.model for _, r in pd.read_csv(paths.REAL_WEATHER_TEMP_CSV).iterrows()}

wcache = weather_real.fetch_temp_humidity_track(paths, TE_REAL)
print(f"{len(wcache)}/{len(TE_REAL)} real buildings have published temperature/humidity")

for name in [m for m in config.ALL_MODELS if m not in ("xgboost", "lightgbm")]:
    if name in rwdone:
        print(f"skip {name} (+weather real-eval)")
        continue
    t0 = time.time()
    try:
        base = models.build(name, L=config.L, H=config.H, n_time=TR["Ft"], n_weather=TR["Fw"],
                             use_weather=True, **models.MODEL_KW.get(name, {})).to(DEV)
        if models.count_params(base) > 0:
            sd_path = f"{paths.WEIGHTS_DIR}/{name}_B.pt"
            if not os.path.exists(sd_path):
                print(f"  {name}: no +weather checkpoint yet, skip")
                continue
            base.load_state_dict(torch.load(sd_path, map_location=DEV))
        print(f"+weather real-eval {name}...", flush=True)
        res = train.evaluate_real(base, TE_REAL, DEV, load_transform, weather_cache=wcache,
                                   weather_transforms=weather_transforms,
                                   amp=config.USE_AMP and name not in config.RNN_MODELS, stride=24)
        os.makedirs(paths.PERBUILDING_REAL_WEATHER_TEMP_DIR, exist_ok=True)
        pickle.dump({k: res[k] for k in ("_nrmse", "_nmae", "_crps", "_is_res") if k in res},
                    open(f"{paths.PERBUILDING_REAL_WEATHER_TEMP_DIR}/{name}.pkl", "wb"))
        row = {"model": name,
               "com_nrmse": round(res.get("Com NRMSE", float("nan")), 3),
               "res_nrmse": round(res.get("Res NRMSE", float("nan")), 3),
               "com_nmae": round(res.get("Com NMAE", float("nan")), 3) if res.get("Com NMAE") == res.get("Com NMAE") else "",
               "res_nmae": round(res.get("Res NMAE", float("nan")), 3) if res.get("Res NMAE") == res.get("Res NMAE") else "",
               "com_crps": round(res.get("Com CRPS", float("nan")), 3) if res.get("Com CRPS") == res.get("Com CRPS") else "",
               "res_crps": round(res.get("Res CRPS", float("nan")), 3) if res.get("Res CRPS") == res.get("Res CRPS") else "",
               "n_buildings": res.get("n_buildings", 0), "n_weather_matched": res.get("n_weather_matched", 0),
               "sec": round(time.time() - t0)}
        csv.DictWriter(open(paths.REAL_WEATHER_TEMP_CSV, "a", newline=""), RWFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (matched={row['n_weather_matched']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
print("DONE ->", paths.REAL_WEATHER_TEMP_CSV)


## Final comparison: no-weather vs +weather on real buildings

Side-by-side Com NRMSE, matched by model, on the subset of real buildings both tracks actually
evaluated (only buildings with published weather appear in the +weather column — see cell 9's
match count above for how many that is per dataset).


In [ ]:
# === 10) Final side-by-side: does +weather (temp/humidity) help on real buildings? ===
real_nw = pd.read_csv(paths.REAL_CSV).set_index("model")
real_w = pd.read_csv(paths.REAL_WEATHER_TEMP_CSV).set_index("model")
print(f"{'model':24s} {'no-weather':>12s} {'+weather':>12s} {'delta':>8s} {'matched':>8s}")
for name in config.ALL_MODELS:
    if name in real_nw.index and name in real_w.index:
        a = real_nw.loc[name, "com_nrmse"]
        b = real_w.loc[name, "com_nrmse"]
        matched = real_w.loc[name, "n_weather_matched"]
        tag = "  (weather helps)" if b < a else ""
        print(f"{name:24s} {a:12.3f} {b:12.3f} {b - a:+8.3f} {matched:8.0f}{tag}")
